# EasyOCR benchmark (simple)

Measures **speed**, **Type I / II**, and **jersey label** checks (`EXPECTED_JERSEY`).

Images: `images/base/` (10) and `images/with_number/` (≥1, all should be jersey **5**). Run all cells.

In [ ]:
import random
import re
import time
from pathlib import Path
from statistics import mean

import easyocr
import numpy as np
from PIL import Image
try:
    RESAMPLE = Image.Resampling.LANCZOS
except AttributeError:
    RESAMPLE = Image.LANCZOS  # Pillow < 9.1

ROOT = Path.cwd()
BASE_DIR = ROOT / "images" / "base"
NUMBER_DIR = ROOT / "images" / "with_number"
EXT = {".png", ".jpg", ".jpeg", ".webp", ".bmp"}

N_SLOTS = 10
N_TRIALS = 50
P_HAS_NUMBER = 0.3
RANDOM_SEED = 42
WARMUP = True
EXPECTED_JERSEY = "5"

IMAGE_SCALE = 4
OCR_KWARGS = {
    "allowlist": "0123456789",
    "mag_ratio": 1.5,
    "contrast_ths": 0.05,
    "adjust_contrast": 0.7,
}
MIN_CONFIDENCE = 0.2
USE_GPU = None


def list_images(folder: Path) -> list[Path]:
    return sorted(
        [p for p in folder.iterdir() if p.is_file() and p.suffix.lower() in EXT],
        key=lambda p: p.name.lower(),
    )


def load_rgb(path: Path) -> np.ndarray:
    im = Image.open(path).convert("RGB")
    if IMAGE_SCALE != 1:
        w, h = im.size
        im = im.resize((w * IMAGE_SCALE, h * IMAGE_SCALE), RESAMPLE)
    return np.array(im)


def parse_jersey_label(ocr_text: str) -> str:
    digits = re.sub(r"\D", "", ocr_text)
    return digits if digits else "(none)"


def read_frame(reader, path: Path) -> tuple[str, float, str]:
    t0 = time.perf_counter()
    raw = reader.readtext(load_rgb(path), detail=1, paragraph=False, **OCR_KWARGS)
    ms = (time.perf_counter() - t0) * 1000.0
    parts = [
        str(x[1]).strip()
        for x in raw
        if float(x[2]) >= MIN_CONFIDENCE and str(x[1]).strip()
    ]
    text = " | ".join(parts) if parts else ""
    return text, ms, parse_jersey_label(text)


def label_error(expected: str, predicted: str, should_have_number: bool) -> str | None:
    if should_have_number:
        if predicted == "(none)":
            return "none"
        if predicted != expected:
            return "wrong"
        return None
    if predicted != "(none)":
        return "false_positive"
    return None


def build_trial(rng, bases: list[Path], numbers: list[Path]):
    paths = list(bases)
    injected = rng.random() < P_HAS_NUMBER
    slot = None
    if injected:
        slot = rng.randrange(N_SLOTS)
        paths[slot] = rng.choice(numbers)
    return paths, injected, slot


In [ ]:
bases = list_images(BASE_DIR)
numbers = list_images(NUMBER_DIR)
if len(bases) != N_SLOTS:
    raise SystemExit(f"Need {N_SLOTS} images in {BASE_DIR}, found {len(bases)}")
if not numbers:
    raise SystemExit(f"Need at least 1 image in {NUMBER_DIR}")

gpu = USE_GPU
if gpu is None:
    try:
        import torch
        gpu = torch.cuda.is_available()
    except ImportError:
        gpu = False

print("Loading EasyOCR...")
t0 = time.perf_counter()
reader = easyocr.Reader(["en"], gpu=gpu, verbose=False)
init_s = time.perf_counter() - t0
print(f"Reader ready in {init_s:.1f}s (gpu={gpu})")

if WARMUP:
    read_frame(reader, bases[0])

number_names = {p.name for p in numbers}
wrong_labels: list[dict] = []

print(f"\nPreflight with_number/ (expected jersey = {EXPECTED_JERSEY}):")
for path in numbers:
    text, _, predicted = read_frame(reader, path)
    err = label_error(EXPECTED_JERSEY, predicted, should_have_number=True)
    status = "OK" if err is None else err.upper()
    print(f"  [{status:16s}]  {path.name:40s}  got {predicted}")
    if err:
        wrong_labels.append({
            "image": path.name, "trial": "preflight", "slot": "-",
            "expected": EXPECTED_JERSEY, "predicted": predicted,
            "ocr_text": text or "(none)", "reason": err,
        })

rng = random.Random(RANDOM_SEED)
frame_ms: list[float] = []
batch_ms: list[float] = []
tp = fp = fn = tn = 0
trials_with_number = 0
trials_without_number = 0

for trial in range(N_TRIALS):
    paths, injected, inject_slot = build_trial(rng, bases, numbers)
    trials_with_number += injected
    trials_without_number += not injected
    t_batch = time.perf_counter()
    for slot, path in enumerate(paths):
        text, ms, predicted = read_frame(reader, path)
        frame_ms.append(ms)
        is_number_slot = injected and slot == inject_slot
        err = label_error(EXPECTED_JERSEY, predicted, should_have_number=is_number_slot)
        if is_number_slot and predicted == EXPECTED_JERSEY:
            tp += 1
        elif is_number_slot:
            fn += 1
        elif predicted != "(none)":
            fp += 1
        else:
            tn += 1
        if err:
            wrong_labels.append({
                "image": path.name, "trial": trial, "slot": slot,
                "expected": EXPECTED_JERSEY if is_number_slot else "(none)",
                "predicted": predicted, "ocr_text": text or "(none)", "reason": err,
            })
    batch_ms.append((time.perf_counter() - t_batch) * 1000.0)

n_number_slots = tp + fn
n_plain_slots = fp + tn
type_i_pct = 100.0 * fp / n_plain_slots if n_plain_slots else 0.0
type_ii_pct = 100.0 * fn / n_number_slots if n_number_slots else 0.0
label_miss = sum(1 for w in wrong_labels if w["reason"] == "none" and w["expected"] == EXPECTED_JERSEY)
label_bad = sum(1 for w in wrong_labels if w["reason"] == "wrong" and w["expected"] == EXPECTED_JERSEY)

print()
print("=" * 50)
print("EASYOCR BENCHMARK")
print("=" * 50)
print(f"trials:              {N_TRIALS}")
print(f"expected jersey:     {EXPECTED_JERSEY}")
print(f"trials w/ number:    {trials_with_number}")
print(f"trials w/o number:   {trials_without_number}")
print(f"image_scale:         {IMAGE_SCALE}  gpu: {gpu}")
print()
print("SPEED")
print(f"  mean per frame:    {mean(frame_ms):.1f} ms")
print(f"  mean per batch:    {mean(batch_ms):.1f} ms  ({mean(batch_ms)/1000:.2f} s)")
print(f"  batch fps:         {1000.0 * N_SLOTS / mean(batch_ms):.2f}")
print()
print("ERRORS")
print(f"  type I  false positive:  {fp:4d}  ({type_i_pct:.1f}%)")
print(f"  type II false negative:    {fn:4d}  ({type_ii_pct:.1f}% of number slots)")
print(f"  true positive:             {tp:4d}")
print(f"  true negative:             {tn:4d}")
print()
print(f"LABEL (expected {EXPECTED_JERSEY})")
if n_number_slots:
    print(f"  correct on number slots:   {tp:4d}  ({100 * tp / n_number_slots:.1f}%)")
print(f"  wrong digit (not {EXPECTED_JERSEY}):   {label_bad:4d}")
print(f"  no digit (none):             {label_miss:4d}")
print()
print("WRONG LABELS (every failure)")
if not wrong_labels:
    print("  (none)")
else:
    for w in wrong_labels:
        print(f"  {w['image']}  trial={w['trial']} slot={w['slot']}  expected={w['expected']} got={w['predicted']}  [{w['reason']}]  raw=\"{w['ocr_text']}\"")
    print()
    print("NUMBER POOL — wrong at least once")
    for name in sorted(number_names):
        fails = [w for w in wrong_labels if w["image"] == name]
        if not fails:
            continue
        preds = sorted({w["predicted"] for w in fails})
        reasons = sorted({w["reason"] for w in fails})
        print(f"  {name}  got={','.join(preds)}  reasons={','.join(reasons)}")
print("=" * 50)